# Task 3 Lifecycle Operations Notebook

This notebook is Task 3 only.

Use it after you have a trained model.pth artifact (from local training or Colab export).

## 1) Setup

Run this notebook from the AAI repository environment where Django dependencies are installed.

In [ ]:
import os
import json
import uuid
from pathlib import Path

import django
from django.conf import settings
from django.test import Client
from django.core.files.uploadedfile import SimpleUploadedFile

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'ai_service.settings')
django.setup()
settings.ALLOWED_HOSTS.append('testserver')

client = Client()
MODEL_NAME = 'produce-quality'
print('Django ready')

## 2) Select Artifact

Option A: set ARTIFACT_PATH_OVERRIDE manually (for a Colab-exported file copied locally).

Option B: leave it empty to auto-pick the latest semantic version from models/produce-quality/*/artifacts/model.pth.

In [ ]:
ARTIFACT_PATH_OVERRIDE = ''  # e.g. r'C:/path/to/model.pth'

def parse_semver(name: str):
    parts = name.split('.')
    if len(parts) != 3:
        return None
    try:
        return tuple(int(x) for x in parts)
    except ValueError:
        return None

if ARTIFACT_PATH_OVERRIDE.strip():
    artifact_path = Path(ARTIFACT_PATH_OVERRIDE).expanduser().resolve()
else:
    model_root = Path(settings.MODEL_ROOT) / MODEL_NAME
    versions = []
    for child in model_root.iterdir() if model_root.exists() else []:
        if child.is_dir():
            parsed = parse_semver(child.name)
            if parsed is not None:
                versions.append((parsed, child.name))

    if not versions:
        raise RuntimeError('No semantic model versions found. Train model first or set ARTIFACT_PATH_OVERRIDE.')

    latest_version = sorted(versions, key=lambda x: x[0])[-1][1]
    artifact_path = model_root / latest_version / 'artifacts' / 'model.pth'

if not artifact_path.exists():
    raise RuntimeError(f'Artifact not found: {artifact_path}')

print('Using artifact:', artifact_path)

In [ ]:
# Upload + activate with dynamic lifecycle version
upload_version = f'qa-{uuid.uuid4().hex[:8]}'
payload_bytes = artifact_path.read_bytes()
uploaded = SimpleUploadedFile('model.pth', payload_bytes, content_type='application/octet-stream')

upload_response = client.post('/api/task3/models/upload/', {
    'model_name': MODEL_NAME,
    'model_version': upload_version,
    'framework': 'pytorch',
    'artifact': uploaded,
})

activate_response = client.post('/api/task3/models/activate/', {
    'model_name': MODEL_NAME,
    'model_version': upload_version,
})

print('Upload status:', upload_response.status_code)
print('Upload body:', upload_response.content.decode())
print('Activate status:', activate_response.status_code)
print('Activate body:', activate_response.content.decode())
print('Activated lifecycle version:', upload_version)

In [ ]:
# Verify lifecycle list and active version
list_response = client.get('/api/task3/models/')
print('List status:', list_response.status_code)
body = list_response.json()
active = [x for x in body.get('results', []) if x.get('model_name') == MODEL_NAME and x.get('is_active')]
print('Active entry:', json.dumps(active[:1], indent=2))

In [ ]:
# Optional: verify Task 2 inference uses activated model
from io import BytesIO
from PIL import Image

img = BytesIO()
Image.new('RGB', (64, 64), (120, 180, 60)).save(img, format='PNG')
img.seek(0)
sample = SimpleUploadedFile('sample.png', img.read(), content_type='image/png')

predict_response = client.post('/api/task2/predict/', {
    'producer_id': 1,
    'image': sample,
})

print('Predict status:', predict_response.status_code)
predict_json = predict_response.json()
print('model_version_used:', predict_json.get('model_version_used'))
print('mode_note:', (predict_json.get('explanation_payload') or {}).get('note'))